# Writing a model once for three backends

Most ampere fits start with a `Model` subclass of your own. The
[quickstart](quickstart.ipynb) writes one in numpy, and it runs on the
reference path: emcee, zeus, dynesty and simulation-based inference. The
gradient-based engines — NUTS, variational inference, gradient optimisers —
need the model on a differentiable backend, jax or torch, and a model written
in numpy cannot simply be handed to them.

This guide shows how to write the model **once**, against the small array
namespace `ampere.core.ArrayOps`, on the base class
`ampere.core.PortableModel`, so that the same source runs on numpy, jax and
torch. It takes the quickstart's straight line, shows the two lines that pin
it to numpy, rewrites it on the portable base, fits it with emcee on numpy
and with NUTS on jax, and ends with when you do not need any of this.

In [ ]:
import inspect
import sys
import time
from pathlib import Path

import numpy as np
from ampere.results import summary

start = time.perf_counter()


def find_repository_root():
    for directory in [Path.cwd(), *Path.cwd().parents]:
        if (directory / "examples" / "portable_model" / "model.py").exists():
            return directory
    raise FileNotFoundError(
        "examples/portable_model not found in the working directory or any parent. "
        "Run this notebook from inside a clone of the ampere repository."
    )


ROOT = find_repository_root()
sys.path.insert(0, str(ROOT))

from examples.portable_model import fit  # noqa: E402
from examples.portable_model import model as portable  # noqa: E402

The guide's model and its fit live in `examples/portable_model`, and the
cells below show their source with `inspect.getsource` rather than a copy of
it: there is one source of truth, and it is the one the conformance battery
holds to a numpy oracle and to itself across all three backends
(`tests/conformance/test_portable_model.py`). As in the quickstart, the
notebook searches upwards from its own directory for the repository.

## The quickstart's model, and the two lines that pin it to numpy

Here is the quickstart's `LinearModel`, unchanged except for two comments.

In [ ]:
from ampere.core import Model, ModelResult, Parameter, Spectrum
import astropy.units as u
import scipy.stats as st


class QuickstartLinearModel(Model):
    """F_nu = slope * lambda + intercept, in Jy, on the channel "sed"."""

    def __init__(self, wavelength):
        self.register_buffer("wavelength", np.asarray(wavelength, dtype=float), unit=u.um)
        self.register_parameter(Parameter("slope", st.uniform(-10, 20)))
        self.register_parameter(Parameter("intercept", st.uniform(-10, 20)))
        self._template = None

    def compile_for(self, requirements):
        asked = requirements.get("sed")
        if asked is not None and "spectral_axis" in asked:
            grid = np.asarray(asked["spectral_axis"].coordinates(), dtype=float)
            self._template = Spectrum(grid * u.um, np.zeros(grid.size), unit=u.Jy)
        return self

    def evaluate(self, **values):
        ctx = self.context(values)
        if self._template is not None:
            grid = self._template.spectral_axis.values
        else:
            grid = np.asarray(ctx["wavelength"], dtype=float)  # <- pins it to numpy (1)
        flux = float(ctx["slope"]) * grid + float(ctx["intercept"])  # <- pins it to numpy (2)
        spectrum = (
            self._template.with_values(flux)
            if self._template is not None
            else Spectrum(grid * u.um, flux, unit=u.Jy)
        )
        return ModelResult({"sed": spectrum})

On the reference path `ctx["slope"]` is a Python float, and both casts are
harmless. On jax it is a *tracer* — the placeholder jax differentiates
through — and `float(...)` on a tracer raises; on torch it is a tensor
carrying the gradient, and `float(...)` silently drops it, so NUTS would see
a flat density. `np.asarray` on the grid has the same problem in reverse: the
arithmetic must happen in the backend's own array type. Every numpy-only
model has some version of these two lines.

## The same model on `PortableModel`

`ampere.core.PortableModel` takes care of everything except the physics: the
`wavelength` buffer and the channel names, the grid negotiation in
`compile_for`, the native surface (`native_grid`, `native_flux`) the jax and
torch paths call, and `evaluate`, which converts to numpy only at the
container boundary. You write `__init__` (the declaration) and one method,
`_flux(grid, context)`:

In [ ]:
print(inspect.getsource(portable.LinearModel))

In `_flux`, `grid` is already the backend's array, and every parameter in
`context` has been passed through `self.ops.asarray`, so a float, a tracer
and a tensor all arrive as the backend's array; the casts are gone. Anything
beyond ordinary arithmetic goes through `self.ops`, the backend's
`ArrayOps`: `exp`, `log`, `log10`, `sqrt`, `power`, `where`, `clip`,
`interp`, `cumsum`, `trapezoid`, `sum`, `stack` and the rest. Two rules keep
`_flux` portable: never call `np.` on something that depends on a
parameter, and never branch on a parameter's value (`if context["slope"] >
0:`) — use `self.ops.where` instead.

The native twins are one line each. The twin base comes **first**, so that its
namespace and capability flags win; the `__init__` above calls
`super().__init__(...)`, which is what lets the twin base run its own checks
(jax's float64 policy) before yours:

In [ ]:
for name in ("model_jax.py", "model_torch.py"):
    source = (ROOT / "examples" / "portable_model" / name).read_text()
    print(source[source.index("class ") :])

The twins' modules are shown as text rather than imported, because this
page is built without jax or torch installed; importing
`examples.portable_model` itself needs neither.

## Fitting it on numpy, with emcee

The problem is the quickstart's, reduced to the WISE W1 / MIPS 70 catalogue
and the IRS short-low chunk with its calibration factor. The function that
builds it takes the backend's **name**, and the only things that change with
it are the model twin and the module the instrument steps and noise model
come from:

In [ ]:
print(inspect.getsource(fit.build_problem))
print(inspect.getsource(fit._instruments))

In [ ]:
problem = fit.build_problem("reference")
t0 = time.perf_counter()
run_emcee = fit.fit_emcee(problem)
time_emcee = time.perf_counter() - t0
print(f"emcee on numpy (24 walkers, 1000 steps, 500 burn-in): {time_emcee:.1f} s")
summary(run_emcee)

emcee starts from the optimiser's mode (the default since W7.12). At this
budget the convergence check above may still warn: slope and intercept are
strongly correlated, which an ensemble sampler explores slowly. That
correlation is exactly where a gradient-based sampler earns its keep.

## The same problem on jax, under NUTS

Composing the problem on jax is the same call with `"jax"`: the jax twin,
`ampere.backends.jax`'s `SyntheticPhotometry`, `Resample`,
`CalibrationScale` and `IndependentNoise`, and
`ampere.backends.jax.configure_x64()` first — jax's float64 switch is
process-global and ampere never flips it for you, so `fit.backend_module`
calls it before any jax work. NUTS then runs on the realised, differentiable
density.

The documentation is built without jax, so the next two cells run only when
jax is installed; otherwise they say they were skipped, and the result from
a run with jax is quoted below them.

In [ ]:
try:
    import jax  # noqa: F401

    HAVE_JAX = True
except ImportError:
    HAVE_JAX = False
print(f"jax available: {HAVE_JAX}")

In [ ]:
time_nuts = None
if HAVE_JAX:
    problem_jax = fit.build_problem("jax")
    t0 = time.perf_counter()
    run_nuts = fit.fit_nuts(problem_jax)  # 2 chains, 500 warm-up and 500 draws each
    time_nuts = time.perf_counter() - t0
    print(f"NUTS on jax: {time_nuts:.1f} s")
    display(summary(run_nuts))
else:
    print("jax is not installed in this build: the NUTS cells were skipped (result quoted below).")

**The NUTS result, quoted.** Run on 2026-10-08 at commit `10e609e` in the `jax` pixi environment (jax 0.11.1, numpyro 0.21.0, CPU), with exactly the cell above — `fit.fit_nuts(fit.build_problem("jax"))`, two chains of 500 warm-up and 500 draws — in 10.4 s of sampling:

| | mean | sd | eti89_lb | eti89_ub | ess_bulk | ess_tail | r_hat |
|---|---|---|---|---|---|---|---|
| `model.slope` | 1.02 | 0.028 | 0.97 | 1.1 | 239 | 227 | 1.01 |
| `model.intercept` | 0.80 | 0.23 | 0.46 | 1.2 | 225 | 236 | 1.01 |
| `sl.instrument.calibration_scale.scale` | 1.0001 | 0.0025 | 1.0 | 1.0 | 726 | 602 | 1.00 |

`tests/examples/test_portable_model.py` runs the same function at the same budget on CI's jax leg, and checks that it recovers the truth (slope 1, intercept 1) within five posterior standard deviations.

**The two posteriors agree.** emcee on numpy, at the budget above, gives slope 1.019 ± 0.028 and intercept 0.81 ± 0.22; NUTS on jax gives 1.02 ± 0.028 and 0.80 ± 0.23, from the same model source, the same data and the same priors. Both cover the truth; NUTS reaches an R-hat of 1.01 in about 10 s where emcee is still at about 1.08 after 24 000 evaluations. The cell below prints the means of whichever runs this build made.

In [ ]:
def posterior_means(run):
    return {
        name: float(np.asarray(run["posterior"][f"model.{name}"]).mean())
        for name in ("slope", "intercept")
    }


means = {"emcee (numpy)": posterior_means(run_emcee)}
if HAVE_JAX:
    means["NUTS (jax)"] = posterior_means(run_nuts)
for label, values in means.items():
    print(f"{label:>14}: " + ", ".join(f"{k} = {v:.3f}" for k, v in values.items()))

## When you do not need this

* **A curated astropy model.** If your model is one of the curated
  `astropy.modeling` classes (or a compound of them), `from_astropy` already
  does all of the above: `ampere.core.from_astropy` for the reference path,
  and `ampere.backends.jax.from_astropy` / `ampere.backends.torch.from_astropy`
  for a differentiable one built from the same declaration. See
  [the astropy page](../astropy.rst).
* **When to re-declare instead.** The package's own rule
  ([interferometry](../interferometry.rst) §8) is: *when the declaration is
  the expensive, dangerous half and the arithmetic the cheap half, inherit;
  when the declaration is thin and the arithmetic is the whole of the step,
  re-declare; when both are cheap, inherit anyway*, for one pattern per
  codebase. `PortableModel` is the inheriting pattern made the default for a
  model of your own. Re-declare per backend when:
  * the model needs **dtype or device plumbing** — a per-instance `device=`
    or `dtype=`, `.to(...)`, an accelerator. `PortableModel` keeps its arrays
    on the CPU in float64; the shipped `BlackBody`, `ModifiedBlackBody` and
    `PowerLaw` re-declare their arithmetic on each backend's own spectral
    base for exactly this reason;
  * the model calls a **compiled extension** or an external code (a radiative
    transfer code, a Fortran library): it cannot be traced at all, belongs on
    the reference path, and is fitted with the gradient-free engines or
    simulation-based inference;
  * the **arithmetic is the whole of the model** and is better written in each
    library's own idiom (a custom kernel, an FFT layout, a hand-written
    gradient).

In [ ]:
elapsed = time.perf_counter() - start
nuts_text = ", NUTS skipped (no jax)" if time_nuts is None else f", NUTS on jax {time_nuts:.1f} s"
print(f"whole notebook: {elapsed:.1f} s (emcee on numpy {time_emcee:.1f} s{nuts_text})")

**Budget and wall time.** emcee used 24 walkers for 1000 steps with the
first 500 discarded (24 000 likelihood evaluations); NUTS used two chains of
500 warm-up and 500 draws. In the `dev` pixi environment (no jax) the emcee
fit took between 23 and 45 s over three runs, and the whole notebook under a minute; in the `jax`
environment the NUTS fit took about 10 s. The cell above prints the
figures for the run you are looking at.